# Notebook de creación de features

In [92]:
from configuraciones import *
import Utils.map_loader as ml

## Features del Modelo nnELM

1. Información Bottom-Up
2. Similitud del Objetivo (Target Similarity)
3. Evidencia Visual
4. Competencia Foveal-Periférica
5. Probabilidad Posterior
6. Entropía del Prior
7. Entropía Global del Posterior
8. Ganancia de Información
9. Gap de Entropía con el Óptimo
10. Distancia con la Decisión del Modelo
11. Competencia en Matriz de Entropías (Ratio)
12. Competencia en Matriz de Entropías (Densidad de Candidatos)

## Contexto del Dataset

| Dimensión | Valor |
|---|---|
| **Sujetos** | 44 (S101–S119 + et_XXXXXX) |
| **Imágenes** | 102 (escenas naturales HSEM) |
| **Fijaciones totales** | ~43 000 |
| **Target encontrado** | ~67 % de los trials |
| **MSS disponibles** | 1, 2, 4 |
| **Grilla del modelo** | 24 × 32 celdas (cell\_size = 32 px, espacio 768 × 1024) |

---

## Taxonomía de Features (25 en total)

| # | Feature(s) | Fuente | Tipo | NaN | Hipótesis EEG |
|---|---|---|---|---|---|
| 1 | `saliencia_media`, `saliencia_mediana`, `saliencia_pixel` | DeepGaze II | Escalar/fijación | No | N200/P1 — atención bottom-up |
| 2 | `similitud_media`, `similitud_mediana`, `similitud_pixel` | ResNeXt101 | Escalar/fijación | No | N170/N250 — template matching |
| 3 | `evidencia_visual` | `visual_evidence[t, y, x]` | Escalar/fijación | No | N2pc — acumulación de evidencia |
| 4 | `competencia_fp_var`, `competencia_fp_media`, `competencia_fp_max` | `max(W·d², W·d')` global | Escalar/fijación | No | Carga cognitiva — foveal vs periférico |
| 5 | `posterior` | `posterior[t, y, x]` | Escalar/fijación | No | P300 — probabilidad de target |
| 6 | `entropia_prior_saliencia`, `entropia_prior_modelo` | Posterior en t=0 | Escalar/trial | No | Carga inicial — constante por imagen |
| 7 | `entropia_posterior` | `−∑ p·log(p)` del posterior | Escalar/fijación | No | Alpha power — incertidumbre global |
| 8 | `ganancia_informacion` | `KL(P_t ‖ P_{t−1})` | Escalar/fijación | Sí (t=0) | P300 — sorpresa informacional |
| 9 | `eig_optimo`, `eig_en_fijacion`, `gap_entropia` | `expected_ig_map` | Escalar/fijación | Sí (última fix) | Sub-optimalidad — costo de decisión |
| 10 | `distancia_grilla`, `distancia_pixels` | argmax EIG map | Escalar/fijación | Sí (última fix) | Distancia espacial al óptimo del modelo |
| 11 | `competencia_ratio_mayor`, `competencia_ratio_segundo_mayor`, `competencia_ratio` | 1°/2° max del EIG map | Escalar/fijación | No | Ambigüedad en la decisión del modelo |
| 12 | `competencia_densidad`, `competencia_densidad_norm` | Celdas ≥ 95% del max EIG | Escalar/fijación | No | Dispersión del EIG map |

**NaN estructurales:**
- `ganancia_informacion`: `None` en la fijación 0 (sin referencia anterior).
- `gap_entropia`, `eig_en_fijacion`, `distancia_*`: `None` en la **última** fijación del trial (no hay t+1).

---

## Estructura de los resultados

Guardo las features en un JSON en Features/Imagen/Sujeto. Es decir, para cada imagen y cada sujeto tengo un JSON que me dice, para cada fijacion, el valor de cada una de las features

### Estructura del JSON de features

```
Features/
└── <imagen_stem>/          # ej. cmp_building_020_cat_007
    └── <sujeto>/           # ej. et_179678
        └── features.json
```

```json
{
  "imagen":       "cmp_building_020_cat_007.jpg",
  "sujeto":       "et_179678",
  "target_stim":  "cat_007.jpg",
  "target_found": true,
  "memory_set":   ["cat_007.jpg", "dog_003.jpg", "bird_001.jpg", "horse_002.jpg"],
  "fijaciones": [
    {
      "fixation": 0,
      "fix_y": 11,  "fix_x": 18,            // celda en grilla 24×32
      "saliencia_media": 0.42,  "saliencia_mediana": 0.39,  "saliencia_pixel": 0.51,
      "similitud_media": 0.18,  "similitud_mediana": 0.15,  "similitud_pixel": 0.21,
      "evidencia_visual": 0.033,
      "competencia_fp_var": 0.0012,  "competencia_fp_media": 0.008,  "competencia_fp_max": 0.14,
      "posterior": 0.0011,
      "entropia_prior_saliencia": 7.82,  "entropia_prior_modelo": 7.95,
      "entropia_posterior": 7.91,
      "ganancia_informacion": null,       // null solo en fijación 0
      "eig_optimo": 0.031,  "eig_en_fijacion": 0.024,  "gap_entropia": 0.007,
      "modelo_fix_y": 9,  "modelo_fix_x": 20,
      "distancia_grilla": 2.83,  "distancia_pixels": 90.5,
      "competencia_ratio_mayor": 0.031,  "competencia_ratio_segundo_mayor": 0.029,  "competencia_ratio": 0.935,
      "competencia_densidad": 12,  "competencia_densidad_norm": 0.0156
    },
    // ...
    {
      "fixation": 9,   // última fijación: gap/distancia/eig_en_fijacion son null
      "ganancia_informacion": 0.0041,
      "eig_optimo": 0.028,  "eig_en_fijacion": null,  "gap_entropia": null,
      "distancia_grilla": null,  "distancia_pixels": null
    }
  ]
}
```

**Notas de coordinadas:**
- `fix_y`, `fix_x` — índices de celda en la grilla 24 × 32 (fuente: NPZ del modelo).
- Las coordenadas de píxel exacto del sujeto vienen del JSON de eye-tracker (`X`, `Y` en espacio imagen 1280 × 1024) y se proyectan al espacio del mapa (768 × 1024) para extraer `*_pixel`.

In [93]:
FEATURES_PATH = '../Features'

def cargar_o_crear_json(path: str, defaults: dict) -> dict:
    """Carga el JSON si existe, si no lo crea con los defaults."""
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return defaults

In [94]:
# Lista de sujetos
path_sujetos = '../Human_Maps/HSUBANOTT'
sujetos = sorted(os.listdir(path_sujetos))
print('Sujetos disponibles:', sujetos)
print('Número de sujetos:', len(sujetos))

Sujetos disponibles: ['S101', 'S102', 'S103', 'S105', 'S106', 'S107', 'S108', 'S109', 'S110', 'S111', 'S112', 'S113', 'S114', 'S115', 'S116', 'S117', 'S118', 'S119', 'et_117969', 'et_123082_2', 'et_179678', 'et_200877', 'et_244108', 'et_244389', 'et_248301', 'et_286470', 'et_305138', 'et_370500', 'et_373490', 'et_389622', 'et_400297', 'et_419760', 'et_501896', 'et_533569', 'et_576470', 'et_601753', 'et_619958', 'et_629959_2', 'et_664304_2', 'et_677251', 'et_712871', 'et_848643', 'et_862513', 'et_963607']
Número de sujetos: 44


In [95]:
import shutil

#shutil.rmtree('Features')
print('Carpeta Features eliminada.')

Carpeta Features eliminada.


## 1. Información Bottom-Up

FEATURES:
- saliencia_media: media del mapa de saliencia en los pixeles de la grilla
- saliencia_mediana: mediana del mapa de saliencia en los pixeles de la grilla

In [96]:
# ── Mapa de saliencia: versión CORRECTA + _original (referencia histórica) ──
#
# BUG CORREGIDO: El código anterior usaba screen_width/screen_height (1920×1080)
# para proyectar las coordenadas del eye-tracker al espacio del mapa de saliencia.
# Las coordenadas X,Y del JSON están en espacio de IMAGEN (1280×1024), no de pantalla.
# Usar screen_width produce un sub-muestreo erróneo (factor ≈0.667 en X, ≈0.948 en Y),
# haciendo que el pixel de fijación caiga en una celda DIFERENTE a la que el modelo asignó.
#
# FIX: normalizar con image_width/image_height del propio JSON.
# El grid_y/grid_x del modelo se obtiene como: int(y / image_height * 24) e int(x / image_width * 32).
# Escalando al mapa de saliencia (768×1024): int(y * 768 / image_height) → idéntico resultado.
# Usamos fixations_y/fixations_x del NPZ como fuente de verdad para los límites del bloque 32×32.

for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    print(f'Procesando sujeto: {sujeto}, número de imágenes: {len(imagenes)}')

    with open(f'../Datasets/HSEM/human_scanpaths/{sujeto}_scanpaths.json') as f:
        scanpaths = json.load(f)

    for imagen in imagenes:
        print(f'  Procesando imagen: {imagen}')

        if imagen not in scanpaths:
            print(f'    ⚠️  Sin scanpath para {imagen}, saltando...')
            continue

        mapa_sal = ml.cargar_mapa_saliencia(imagen, como_grilla=False)
        mapas    = ml.cargar_mapas_humanos(sujeto, imagen)

        print(f'    Dimensiones mapa saliencia: {mapa_sal.shape}')

        scanpath    = scanpaths[imagen]

        # ── Dimensiones correctas: espacio de imagen (donde viven X,Y del scanpath) ──
        img_width   = scanpath['image_width']   # 1280
        img_height  = scanpath['image_height']  # 1024

        # ── Dimensiones de pantalla (SOLO para _original, registro del bug previo) ──
        scr_width   = scanpath['screen_width']  # 1920
        scr_height  = scanpath['screen_height'] # 1080

        # Coordenadas del pixel al que mira el sujeto, en espacio de imagen (1280×1024)
        xs_pantalla = scanpath['X']
        ys_pantalla = scanpath['Y']

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [],
        })
        data['fijaciones'] = []  # Reconstruir siempre para evitar desfases

        for i, (grid_y, grid_x, x_p, y_p) in enumerate(zip(
                mapas['fixations_y'], mapas['fixations_x'],
                xs_pantalla, ys_pantalla)):

            # ── VERSIÓN CORRECTA ──────────────────────────────────────────────
            # Proyección: imagen (1280×1024) → mapa de saliencia (1024×768)
            x_img = min(int(x_p * (mapa_sal.shape[1] / img_width)),  mapa_sal.shape[1] - 1)
            y_img = min(int(y_p * (mapa_sal.shape[0] / img_height)), mapa_sal.shape[0] - 1)

            # Bloque 32×32 alineado con el grid del modelo (fuente de verdad: NPZ)
            y_ini = int(grid_y) * 32
            x_ini = int(grid_x) * 32
            y_fin = min(y_ini + 32, mapa_sal.shape[0])
            x_fin = min(x_ini + 32, mapa_sal.shape[1])
            bloque = mapa_sal[y_ini:y_fin, x_ini:x_fin]

            # ── VERSIÓN _original (bug previo: screen_width → registro histórico) ──
            x_img_o = min(int(x_p * (mapa_sal.shape[1] / scr_width)),  mapa_sal.shape[1] - 1)
            y_img_o = min(int(y_p * (mapa_sal.shape[0] / scr_height)), mapa_sal.shape[0] - 1)
            gy_o = y_img_o // 32
            gx_o = x_img_o // 32
            yo_ini = gy_o * 32
            xo_ini = gx_o * 32
            bloque_o = mapa_sal[yo_ini:min(yo_ini + 32, mapa_sal.shape[0]),
                                xo_ini:min(xo_ini + 32, mapa_sal.shape[1])]

            data['fijaciones'].append({
                'fixation':                   i,
                'fix_y':                      int(grid_y),
                'fix_x':                      int(grid_x),
                # Correctas
                'saliencia_media':            float(np.mean(bloque))   if bloque.size   > 0 else 0.0,
                'saliencia_mediana':          float(np.median(bloque)) if bloque.size   > 0 else 0.0,
                'saliencia_pixel':            float(mapa_sal[y_img, x_img]),
                # _original (referencia histórica del bug previo)
                # 'saliencia_media_original':   float(np.mean(bloque_o))   if bloque_o.size > 0 else 0.0,
                # 'saliencia_mediana_original': float(np.median(bloque_o)) if bloque_o.size > 0 else 0.0,
                # 'saliencia_pixel_original':   float(mapa_sal[y_img_o, x_img_o]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)


Procesando sujeto: S101, número de imágenes: 102
  Procesando imagen: cmp_africansavanna_003_wildanimals_003.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_beach_002_person_002.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_beach_003_bird_001.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_building_003_person_001.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_building_006_person_003.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_building_007_person_004.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_building_010_cat_002.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_building_014_person_007.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_building_015_person_008.jpg
    Dimensiones mapa saliencia: (768, 1024)
  Procesando imagen: cmp_building_018_cat_005.jpg
    Dimensiones mapa salienc

## 2. Target Similarity

FEATURES:
- similitud_media: media del mapa de similitud en los pixeles de la grilla
- similitud_mediana: mediana del mapa de similitud en los pixeles de la grilla

In [97]:
# ── Mapa de similitud: versión CORRECTA + _original (referencia histórica) ──
#
# BUG CORREGIDO: mismo error que en saliencia — se usaba screen_width/screen_height
# para calcular similitud_pixel. El bloque 32×32 ya era correcto (desde NPZ),
# pero la extracción a nivel de pixel apuntaba a la posición equivocada.
#
# Nota: similitud_media y similitud_mediana no cambian (el bloque era correcto).
# similitud_pixel y similitud_pixel_original SÍ difieren.

for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    print(f'Procesando sujeto: {sujeto}, número de imágenes: {len(imagenes)}')

    with open(f'../Datasets/HSEM/human_scanpaths/{sujeto}_scanpaths.json') as f:
        scanpaths = json.load(f)

    for imagen in imagenes:
        print(f'  Procesando imagen: {imagen}')

        if imagen not in scanpaths:
            print(f'    ⚠️  Sin scanpath para {imagen}, saltando...')
            continue

        mapas  = ml.cargar_mapas_humanos(sujeto, imagen)
        target = mapas['target_stim']

        if not ml.mapa_similitud_existe(imagen, target):
            print(f'    ⚠️  Sin mapa de similitud para {imagen} / {target}')
            continue

        mapa_sim = ml.cargar_mapa_similitud(imagen, target, como_grilla=False)

        scanpath    = scanpaths[imagen]
        img_width   = scanpath['image_width']
        img_height  = scanpath['image_height']
        scr_width   = scanpath['screen_width']   # solo para _original
        scr_height  = scanpath['screen_height']  # solo para _original
        xs_pantalla = scanpath['X']
        ys_pantalla = scanpath['Y']

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  target,
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        for i, (y, x, x_p, y_p) in enumerate(zip(
                mapas['fixations_y'], mapas['fixations_x'],
                xs_pantalla, ys_pantalla)):

            # Bloque 32×32 desde NPZ (correcto en ambas versiones)
            y_ini = int(y) * 32
            x_ini = int(x) * 32
            y_fin = min(y_ini + 32, mapa_sim.shape[0])
            x_fin = min(x_ini + 32, mapa_sim.shape[1])
            bloque_sim = mapa_sim[y_ini:y_fin, x_ini:x_fin]

            # ── VERSIÓN CORRECTA: pixel exacto con image_width ────────────────
            x_img = min(int(x_p * (mapa_sim.shape[1] / img_width)),  mapa_sim.shape[1] - 1)
            y_img = min(int(y_p * (mapa_sim.shape[0] / img_height)), mapa_sim.shape[0] - 1)

            # ── VERSIÓN _original: pixel con screen_width (bug previo) ────────
            x_img_o = min(int(x_p * (mapa_sim.shape[1] / scr_width)),  mapa_sim.shape[1] - 1)
            y_img_o = min(int(y_p * (mapa_sim.shape[0] / scr_height)), mapa_sim.shape[0] - 1)

            data['fijaciones'][i].update({
                # Correctas (bloque igual en ambas; pixel distinto)
                'similitud_media':            float(np.mean(bloque_sim)),
                'similitud_mediana':          float(np.median(bloque_sim)),
                'similitud_pixel':            float(mapa_sim[y_img, x_img]),
                # _original (registro histórico)
                # 'similitud_media_original':   float(np.mean(bloque_sim)),    # idéntico
                # 'similitud_mediana_original': float(np.median(bloque_sim)),  # idéntico
                # 'similitud_pixel_original':   float(mapa_sim[y_img_o, x_img_o]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)


Procesando sujeto: S101, número de imágenes: 102
  Procesando imagen: cmp_africansavanna_003_wildanimals_003.jpg
  Procesando imagen: cmp_beach_002_person_002.jpg
  Procesando imagen: cmp_beach_003_bird_001.jpg
  Procesando imagen: cmp_building_003_person_001.jpg
  Procesando imagen: cmp_building_006_person_003.jpg
  Procesando imagen: cmp_building_007_person_004.jpg
  Procesando imagen: cmp_building_010_cat_002.jpg
  Procesando imagen: cmp_building_014_person_007.jpg
  Procesando imagen: cmp_building_015_person_008.jpg
  Procesando imagen: cmp_building_018_cat_005.jpg
  Procesando imagen: cmp_building_020_cat_007.jpg
  Procesando imagen: cmp_building_021_cat_008.jpg
  Procesando imagen: cmp_building_022_cat_009.jpg
  Procesando imagen: cmp_building_023_cat_010.jpg
  Procesando imagen: cmp_building_030_person_016.jpg
  Procesando imagen: cmp_building_032_cat_011.jpg
  Procesando imagen: cmp_building_033_cat_012.jpg
  Procesando imagen: cmp_building_039_dog_007.jpg
  Procesando imagen: 

## 3. Evidencia Visual

In [98]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    print(f'Procesando sujeto: {sujeto}, número de imágenes: {len(imagenes)}')

    for imagen in imagenes:
        print(f'  Procesando imagen: {imagen}')
        
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # ve_at_fix: valor de la evidencia visual acumulada en la celda fijada
        ve_at_fix = ml.valor_en_fijacion(mapas['visual_evidence'], mapas)

        for i, (y, x) in enumerate(zip(mapas['fixations_y'], mapas['fixations_x'])):
            data['fijaciones'][i].update({
                'fixation':   i,
                'fix_y':      int(y),
                'fix_x':      int(x),
                'evidencia_visual':  float(ve_at_fix[i]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

Procesando sujeto: S101, número de imágenes: 102
  Procesando imagen: cmp_africansavanna_003_wildanimals_003.jpg
  Procesando imagen: cmp_beach_002_person_002.jpg
  Procesando imagen: cmp_beach_003_bird_001.jpg
  Procesando imagen: cmp_building_003_person_001.jpg
  Procesando imagen: cmp_building_006_person_003.jpg
  Procesando imagen: cmp_building_007_person_004.jpg
  Procesando imagen: cmp_building_010_cat_002.jpg
  Procesando imagen: cmp_building_014_person_007.jpg
  Procesando imagen: cmp_building_015_person_008.jpg
  Procesando imagen: cmp_building_018_cat_005.jpg
  Procesando imagen: cmp_building_020_cat_007.jpg
  Procesando imagen: cmp_building_021_cat_008.jpg
  Procesando imagen: cmp_building_022_cat_009.jpg
  Procesando imagen: cmp_building_023_cat_010.jpg
  Procesando imagen: cmp_building_030_person_016.jpg
  Procesando imagen: cmp_building_032_cat_011.jpg
  Procesando imagen: cmp_building_033_cat_012.jpg
  Procesando imagen: cmp_building_039_dog_007.jpg
  Procesando imagen: 

## 4. Competencia Foveal-Periferica

Guardo la varianza, la media y el máximo de la competencia foveal-periferica

In [100]:
import sys
sys.path.insert(0, '..')

from Model.visualsearch.visibility_map import VisibilityMap
from Model.visualsearch.grid import Grid

image_size = (768, 1024)

grid = Grid(np.array(image_size), cell_size=32) # grid.size() → (24, 32)
sigma = [[4000, 0], [0, 2600]]  # covarianza gaussiana, eje Y y eje X respectivamente

vis = VisibilityMap(image_size, grid, sigma, fovea_exponent=2, peripheral_exponent=1)

for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        fix_ys = mapas['fixations_y']  # coordenadas en grilla (24×32), ya correctas
        fix_xs = mapas['fixations_x']

        for i, (gy, gx) in enumerate(zip(fix_ys, fix_xs)):
            W_t = mapas['visual_evidence'][i]          # shape: (24, 32)

            # Kernels de visibilidad centrados en la fijación actual
            # (igual que en codigo del modelo)
            d2      = vis.at_fixation_fovea((gy, gx))  # d²  — término foveal
            d_prime = vis.at_fixation((gy, gx))         # d'  — término periférico

            # Mapa de competencia foveal-periférica (Eq. 8 del paper)
            foveal_term     = W_t * d2
            peripheral_term = W_t * d_prime
            competition_map = np.maximum(foveal_term, peripheral_term)  # shape: (24, 32)

            data['fijaciones'][i].update({
                'competencia_fp_var':   float(np.var(competition_map)),
                'competencia_fp_media': float(np.mean(competition_map)),
                'competencia_fp_max':   float(np.max(competition_map)),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

In [101]:
# CODIGO ORIGINAL (ERROR: La competencia visual se calculaba como var/mean/max del max entre W y d, no de W)

# for sujeto in sujetos:
#     imagenes = ml.listar_imagenes(sujeto)
#     for imagen in imagenes:
#         mapas = ml.cargar_mapas_humanos(sujeto, imagen)
# 
#         imagen_stem = imagen.replace('.jpg', '')
#         output_dir  = f'Features/{imagen_stem}/{sujeto}'
#         os.makedirs(output_dir, exist_ok=True)
#         json_path   = f'{output_dir}/features.json'
# 
#         data = cargar_o_crear_json(json_path, defaults={
#             'imagen':       imagen,
#             'sujeto':       sujeto,
#             'target_stim':  mapas['target_stim'],
#             'target_found': mapas['target_found'],
#             'memory_set':   mapas['memory_set'],
#             'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
#         })
# 
#         competencia_var    = np.var(mapas['visual_evidence'],  axis=(1, 2))
#         competencia_media  = np.mean(mapas['visual_evidence'], axis=(1, 2))
#         competencia_max    = np.max(mapas['visual_evidence'],  axis=(1, 2))
# 
#         for i in range(len(mapas['fixations_y'])):
#             data['fijaciones'][i].update({
#                 'competencia_fp_var':   float(competencia_var[i]),
#                 'competencia_fp_media': float(competencia_media[i]),
#                 'competencia_fp_max':   float(competencia_max[i]),
#             })
# 
#         with open(json_path, 'w') as f:
#             json.dump(data, f, indent=2)

## 5. Probabilidad Posterior

In [102]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # Valor del posterior en la celda fijada — escalar por fijación
        posterior_at_fix = ml.valor_en_fijacion(mapas['posterior'], mapas)

        for i in range(len(mapas['fixations_y'])):
            data['fijaciones'][i].update({
                'posterior': float(posterior_at_fix[i]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 6. Entropía del Prior

El mapa de saliencia (DeepGaze II) no es una distribución de probabilidad: sus valores están en [0,1] pero no suman 1. Es una función de puntuación que indica relevancia visual relativa de cada celda.
El modelo lo convierte en distribución en dos pasos: primero normaliza por el máximo, luego aplica una transformación afín que garantiza peso mínimo a cada celda. El posterior resultante en t=0 sí es una distribución válida (suma 1) y representa la creencia inicial real del modelo.
Se guardan dos versiones:

entropia_prior_saliencia: entropía del mapa de saliencia crudo normalizado manualmente. Mide la complejidad visual de la escena.
entropia_prior_modelo: entropía del posterior[0], es decir el prior procesado que realmente usa el modelo. Mide la incertidumbre inicial del proceso de búsqueda.

Ambas son constantes a lo largo del trial (el prior no cambia entre fijaciones).

In [103]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas    = ml.cargar_mapas_humanos(sujeto, imagen)
        mapa_sal = ml.cargar_mapa_saliencia(imagen, como_grilla=True)  # (24, 32)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # Opción A: entropía de la saliencia cruda normalizada
        prior_sal  = mapa_sal / mapa_sal.sum()
        entropia_a = float(-np.sum(prior_sal * np.log(prior_sal + 1e-12)))

        # Opción B: entropía del prior real del modelo (posterior en fijación 0)
        prior_mod  = mapas['posterior'][0]
        prior_mod  = prior_mod / prior_mod.sum()
        entropia_b = float(-np.sum(prior_mod * np.log(prior_mod + 1e-12)))

        # Es una feature de la imagen, no de cada fijación — igual valor en todas
        for i in range(len(mapas['fixations_y'])):
            data['fijaciones'][i].update({
                'entropia_prior_saliencia': entropia_a,
                'entropia_prior_modelo':    entropia_b,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 7. Entropía del Posterior

In [104]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # Entropía de Shannon del posterior en cada fijación
        # entropy_map ya tiene -p·log(p) celda a celda — shape (n_fix, 24, 32)
        # Su suma sobre el mapa da la entropía escalar
        entropia_posterior = ml.entropia_escalar(mapas)  # shape (n_fix,)

        for i in range(len(mapas['fixations_y'])):
            data['fijaciones'][i].update({
                'entropia_posterior': float(entropia_posterior[i]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 8. Ganancia de la información

def kl_divergencia_consecutiva(data: dict) -> np.ndarray:
    """
    Divergencia KL entre posteriors consecutivos: KL(P_t || P_{t-1}).

    Mide cuánta información aportó cada fijación al actualizar la creencia.
    Análogo al elm_info_gained del modelo, pero calculado sobre los posteriors reales.

    Shape: (n_fix - 1,)  — la fijación 0 no tiene referencia anterior.
    """
    post = data['posterior']   # (n_fix, 24, 32)
    eps = 1e-12
    return np.sum(post[1:] * np.log((post[1:] + eps) / (post[:-1] + eps)), axis=(1, 2))

In [105]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # KL(P_t || P_{t-1}) — shape (n_fix - 1,)
        kl = ml.kl_divergencia_consecutiva(mapas)

        for i in range(len(mapas['fixations_y'])):
            # La fijación 0 no tiene referencia anterior
            kl_val = float(kl[i - 1]) if i > 0 else None

            data['fijaciones'][i].update({
                'ganancia_informacion': kl_val,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 9. Gap de Entropía con el Óptimo

In [106]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)
        fix_ys  = mapas['fixations_y']
        fix_xs  = mapas['fixations_x']
        n_fix   = len(fix_ys)

        for i in range(n_fix):

            optimo = float(eig_map[i].max())

            if i < n_fix - 1:
                y_sig            = fix_ys[i + 1]
                x_sig            = fix_xs[i + 1]
                eig_en_siguiente = float(eig_map[i, y_sig, x_sig])
                gap              = optimo - eig_en_siguiente
            else:
                eig_en_siguiente = None
                gap              = None

            data['fijaciones'][i].update({
                'eig_optimo':      optimo,
                'eig_en_fijacion': eig_en_siguiente,
                'gap_entropia':    gap,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 10. Distancia con decisión del modelo

In [107]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)
        fix_ys  = mapas['fixations_y']
        fix_xs  = mapas['fixations_x']
        n_fix   = len(fix_ys)

        for i in range(n_fix):

            coords_optimo  = np.unravel_index(eig_map[i].argmax(), eig_map[i].shape)
            y_modelo, x_modelo = coords_optimo

            if i < n_fix - 1:
                y_sig            = fix_ys[i + 1]
                x_sig            = fix_xs[i + 1]
                distancia_grilla = float(np.sqrt((y_sig - y_modelo)**2 + (x_sig - x_modelo)**2))
                distancia_pixels = distancia_grilla * 32
            else:
                distancia_grilla = None
                distancia_pixels = None

            data['fijaciones'][i].update({
                'modelo_fix_y':     int(y_modelo),
                'modelo_fix_x':     int(x_modelo),
                'distancia_grilla': distancia_grilla,
                'distancia_pixels': distancia_pixels,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 11. Competencia en Matriz de Entropias (ratio)

In [108]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)

        for i in range(len(mapas['fixations_y'])):

            # Aplanar y ordenar de mayor a menor
            valores = np.sort(eig_map[i].flatten())[::-1]

            mayor        = float(valores[0])
            segundo_mayor = float(valores[1])

            # Evitar división por cero
            ratio = float(segundo_mayor / mayor) if mayor > 0 else None

            data['fijaciones'][i].update({
                'competencia_ratio_mayor':        mayor,
                'competencia_ratio_segundo_mayor': segundo_mayor,
                'competencia_ratio':              ratio,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 12. Competencia en matriz de entropias (densidad)  

In [109]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)

        for i in range(len(mapas['fixations_y'])):

            mayor   = float(eig_map[i].max())
            umbral  = mayor * 0.95

            # Cantidad de celdas dentro del 5% del máximo
            densidad = int(np.sum(eig_map[i] >= umbral))

            # Versión normalizada: proporción sobre el total de celdas (24*32=768)
            densidad_norm = float(densidad / eig_map[i].size)

            data['fijaciones'][i].update({
                'competencia_densidad':      densidad,
                'competencia_densidad_norm': densidad_norm,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

---
## Verificación y Resumen del Dataset de Features

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew

# ── Cargar todos los features en un DataFrame ─────────────────────────────────
rows = []
for imagen_stem in sorted(os.listdir(FEATURES_PATH)):
    imagen = imagen_stem + '.jpg'
    suj_dir = f'{FEATURES_PATH}/{imagen_stem}'
    if not os.path.isdir(suj_dir):
        continue
    for sujeto in sorted(os.listdir(suj_dir)):
        json_path = f'{suj_dir}/{sujeto}/features.json'
        if not os.path.exists(json_path):
            continue
        with open(json_path) as f:
            data = json.load(f)
        for fij in data['fijaciones']:
            rows.append({
                'imagen': imagen, 'sujeto': sujeto,
                'target_found': data['target_found'],
                'mss': len(data['memory_set']),
                **fij
            })

df = pd.DataFrame(rows)
df['ganancia_informacion'] = pd.to_numeric(df['ganancia_informacion'], errors='coerce')

FEAT_COLS = [c for c in df.columns if c not in
             ['imagen','sujeto','target_found','mss','fixation',
              'fix_y','fix_x','modelo_fix_y','modelo_fix_x']]

# ── Resumen global ────────────────────────────────────────────────────────────
print('=' * 55)
print('          RESUMEN DEL DATASET DE FEATURES')
print('=' * 55)
print(f'  Fijaciones totales   : {len(df):>8,}')
print(f'  Imágenes             : {df["imagen"].nunique():>8}')
print(f'  Sujetos              : {df["sujeto"].nunique():>8}')
tf = df.groupby('target_found').size()
print(f'  Target found=True    : {tf.get(True, 0):>8,}  ({100*tf.get(True,0)/len(df):.1f}%)')
print(f'  Target found=False   : {tf.get(False,0):>8,}  ({100*tf.get(False,0)/len(df):.1f}%)')
mss_counts = df.groupby('mss').size()
for m in sorted(mss_counts.index):
    print(f'  MSS = {m}              : {mss_counts[m]:>8,}  ({100*mss_counts[m]/len(df):.1f}%)')
print(f'  Features extraídas   : {len(FEAT_COLS):>8}')
print()

# ── Tabla de features: escala, NaN, skewness ─────────────────────────────────
print(f'{"Feature":<40s}  {"Media":>10}  {"Std":>10}  {"NaN%":>6}  {"Skew":>7}')
print('-' * 82)
for feat in FEAT_COLS:
    s = df[feat].dropna()
    nan_pct = 100 * df[feat].isna().mean()
    sk = skew(s) if len(s) > 10 else float('nan')
    flag = '  *' if nan_pct > 1 else ''
    print(f'{feat:<40s}  {s.mean():>10.4f}  {s.std():>10.4f}  {nan_pct:>5.1f}%  {sk:>7.2f}{flag}')

print()
print('(*) NaN estructurales esperados: ganancia_informacion[t=0], gap/distancia[última fix)')
print(f'\nFeatures ({len(FEAT_COLS)}): {FEAT_COLS}')